## Open In Colab

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/10_decision_audit/oracle_bottleneck_colab.ipynb)


# Oracle Bottleneck Analysis (Colab)

## Objective
Diagnose where performance bottlenecks come from when using the same candidate sets:

- `current` controller (current thresholded rule with predicted risk)
- `oracle-risk` controller (uses true risk label for safety-first selection)
- `oracle-best` controller (best achievable progress among truly safe candidates)


## Hypotheses Tested

1. Fixed candidate sets can expose whether performance bottlenecks come from signal quality/calibration, controller rule, or candidate quality.
2. RACP-style risk-penalized reranking trades progress for safety as lambda increases.
3. Chance-constrained gating is sensitive to the risk signal and threshold (`tau`).
4. Conformal-style effective thresholding can improve budget adherence vs fixed `tau=0.2`.
5. Effects can differ across planner variants loaded from one or more runs.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

NOTEBOOK_NAME = 'oracle_bottleneck_colab'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for p in (REPO_DIR, f'{REPO_DIR}/src'):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True. Re-run this notebook after runtime restart.')


## Step 4 - Configuration + Run Context


In [ ]:
import numpy as np
import pandas as pd

from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)

cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', cfg.run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_risk_uq_run_context(run_ctx, display_fn=display)

FOCUS_LABEL = str(run_cfg.get('focus_label', 'failure_proxy_h15'))
RISK_BUDGET_TAU = float(run_cfg.get('risk_budget_tau', 0.20))
TAU_SWEEP_VALUES = [round(float(x), 3) for x in list(np.linspace(0.05, 0.80, 16))]
BOOTSTRAP_SAMPLES = int(run_cfg.get('decision_bootstrap_samples', 300))
BOOTSTRAP_SEED = int(run_cfg.get('decision_bootstrap_seed', 17))

# Multi-controller settings.
RACP_LAMBDA_VALUES = run_cfg.get('racp_lambda_values', [0.0, 0.5, 1.0, 2.0, 4.0])
if not isinstance(RACP_LAMBDA_VALUES, (list, tuple)):
    RACP_LAMBDA_VALUES = [0.0, 0.5, 1.0, 2.0, 4.0]
RACP_LAMBDA_VALUES = [float(x) for x in RACP_LAMBDA_VALUES]

PRIMARY_CONTROLLER_VARIANTS = run_cfg.get(
    'controller_risk_variants',
    ['combo:platt', 'combo:raw', 'racp_style:platt', 'ccmpc_style:platt', 'radius_style:platt'],
)
if not isinstance(PRIMARY_CONTROLLER_VARIANTS, (list, tuple)):
    PRIMARY_CONTROLLER_VARIANTS = ['combo:platt', 'combo:raw', 'racp_style:platt', 'ccmpc_style:platt', 'radius_style:platt']
PRIMARY_CONTROLLER_VARIANTS = [str(x).strip() for x in PRIMARY_CONTROLLER_VARIANTS if str(x).strip()]

CONFORMAL_MIN_CAL_ROWS = int(max(30, int(run_cfg.get('conformal_min_cal_rows', 80))))
CONFORMAL_MIN_ACCEPT = int(max(10, int(run_cfg.get('conformal_min_accept_count', 20))))
CONFORMAL_USE_SHIFT_LOCAL = bool(run_cfg.get('conformal_shift_local', False))

print('RACP_LAMBDA_VALUES =', RACP_LAMBDA_VALUES)
print('PRIMARY_CONTROLLER_VARIANTS =', PRIMARY_CONTROLLER_VARIANTS)


## Step 5 - Load Candidate-Level Data (same candidate sets for all controllers)


In [ ]:
from pathlib import Path
from src.workflows import (
    discover_probe_run_prefixes,
    has_existing_miscalibration_probe_artifacts,
    load_existing_miscalibration_probe_bundle,
)

def _parse_prefix_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    text = text.replace(chr(10), ',').replace(';', ',')
    return [p.strip() for p in text.split(',') if p.strip()]

def _normalize_planner_variant(df, run_prefix):
    local = df.copy()
    run_tag = Path(str(run_prefix)).name
    for col in ['planner_variant', 'planner_backend', 'planner_kind', 'planner_used']:
        if col in local.columns:
            s = local[col].astype(str).str.strip()
            if (s != '').any():
                local['planner_variant'] = s.where(s.str.len() > 0, f'run:{run_tag}')
                break
    if 'planner_variant' not in local.columns:
        local['planner_variant'] = f'run:{run_tag}'
    local['planner_variant'] = local['planner_variant'].astype(str)
    local['analysis_run_prefix'] = str(run_prefix)
    local['analysis_run_tag'] = str(run_tag)
    return local

ANALYSIS_RUN_PREFIX_OVERRIDE = str(run_cfg.get('analysis_run_prefix', '')).strip()
ANALYSIS_RUN_PREFIXES = _parse_prefix_list(run_cfg.get('analysis_run_prefixes', []))
MAX_DISCOVERED_RUNS = int(max(1, int(run_cfg.get('max_discovered_probe_runs', 50))))
MAX_RUNS_TO_LOAD = int(max(1, int(run_cfg.get('max_probe_runs_to_load', 8))))

requested_prefixes = []
if ANALYSIS_RUN_PREFIX_OVERRIDE:
    requested_prefixes.append(ANALYSIS_RUN_PREFIX_OVERRIDE)
requested_prefixes.extend(ANALYSIS_RUN_PREFIXES)
if len(requested_prefixes) == 0:
    requested_prefixes.append(str(cfg.run_prefix))

seen = set()
requested_prefixes = [p for p in requested_prefixes if not (p in seen or seen.add(p))]

discovered_runs_df = discover_probe_run_prefixes(PERSIST_ROOT, limit=MAX_DISCOVERED_RUNS)
if not discovered_runs_df.empty:
    display(discovered_runs_df.head(20))

ALLOW_PREVIOUS_RUN_FALLBACK = bool(run_cfg.get('allow_previous_run_fallback', True))
prefixes_to_try = list(requested_prefixes)
if ALLOW_PREVIOUS_RUN_FALLBACK and (not discovered_runs_df.empty):
    for p in discovered_runs_df['run_prefix'].astype(str).tolist():
        if p not in prefixes_to_try:
            prefixes_to_try.append(p)
prefixes_to_try = prefixes_to_try[:MAX_RUNS_TO_LOAD]

frames = []
loaded_prefixes = []
loaded_sources = []
for pfx in prefixes_to_try:
    if not has_existing_miscalibration_probe_artifacts(pfx):
        continue

    cross_signal_pred_path = Path(f'{pfx}_cross_signal_decision_predictions.parquet')
    if cross_signal_pred_path.exists():
        try:
            df_i = pd.read_parquet(cross_signal_pred_path)
            src = 'cross_signal_decision_predictions'
        except Exception:
            df_i = pd.DataFrame()
            src = ''
    else:
        df_i = pd.DataFrame()
        src = ''

    if df_i.empty:
        try:
            bundle = load_existing_miscalibration_probe_bundle(pfx)
            df_i = bundle.predictions_df.copy()
            src = 'miscalibration_probe_predictions'
        except Exception:
            df_i = pd.DataFrame()

    if df_i.empty:
        continue

    df_i = _normalize_planner_variant(df_i, pfx)
    frames.append(df_i)
    loaded_prefixes.append(str(pfx))
    loaded_sources.append(src)

if bool(RUN_ENABLED) and (len(frames) == 0):
    raise FileNotFoundError(
        'No completed miscalibration probe artifacts were found for analysis. '
        'Run decision_audit_artifact_builder_colab.ipynb (or 00_probe/miscalibration_probe_colab.ipynb), or set analysis_run_prefix(es) to existing runs.'
    )

if len(frames) == 0:
    raise RuntimeError('No non-empty candidate dataset loaded for oracle bottleneck notebook.')

data_df = pd.concat(frames, axis=0, ignore_index=True)
analysis_run_prefix = str(loaded_prefixes[0])
AUDIT_OUTPUT_PREFIX = str(cfg.run_prefix)

if 'shift_suite' not in data_df.columns:
    data_df['shift_suite'] = 'nominal_clean'
else:
    data_df['shift_suite'] = data_df['shift_suite'].fillna('nominal_clean').astype(str)

if 'scenario_id' not in data_df.columns:
    data_df['scenario_id'] = np.arange(len(data_df)).astype(str)
data_df['scenario_id'] = data_df['scenario_id'].astype(str)

if 'step_idx' not in data_df.columns:
    data_df['step_idx'] = 0
data_df['step_idx'] = pd.to_numeric(data_df['step_idx'], errors='coerce').fillna(0).astype(int)

# Scenario-level split assumptions.
if 'eval_split' in data_df.columns:
    split_col = data_df['eval_split'].astype(str)
    eval_mask = split_col.isin(['test', 'high_interaction_holdout'])
    cal_mask = split_col.eq('val')
    split_source = 'eval_split test/holdout for eval + val for calibration'
else:
    eval_mask = pd.Series(np.ones((len(data_df),), dtype=bool), index=data_df.index)
    cal_mask = pd.Series(np.ones((len(data_df),), dtype=bool), index=data_df.index)
    split_source = 'eval_split missing; fallback all rows for eval/cal'

eval_df = data_df.loc[eval_mask].copy()
cal_df = data_df.loc[cal_mask].copy()
if eval_df.empty:
    eval_df = data_df.copy()
if cal_df.empty:
    cal_df = data_df.copy()

if FOCUS_LABEL not in eval_df.columns:
    raise ValueError(f'Focus label {FOCUS_LABEL!r} is missing from candidate data.')
progress_col = 'progress_h6' if 'progress_h6' in eval_df.columns else None
if progress_col is None:
    raise ValueError('Required progress column progress_h6 is missing.')

def _clip_prob(x):
    arr = np.asarray(pd.to_numeric(x, errors='coerce'), dtype=float)
    arr = np.where(np.isfinite(arr), arr, 0.5)
    return np.clip(arr, 1e-6, 1.0 - 1e-6)

# Resolve available risk columns by signal:calibration.
available_risk_cols = {}
for c in eval_df.columns:
    if not str(c).startswith('risk_'):
        continue
    cc = str(c)
    if cc.count('_') < 3:
        continue
    # risk_<signal>_<cal>
    parts = cc.split('_')
    signal = '_'.join(parts[1:-1])
    cal = parts[-1]
    key = f'{signal}:{cal}'
    available_risk_cols[key] = cc

# Backward-compatible fallback aliases from probe outputs.
legacy_candidates = {
    'combo:platt': ['planner_risk_combo_platt', 'risk_combo_platt'],
    'combo:raw': ['planner_risk_combo_proxy', 'risk_combo_raw'],
    'top1:platt': ['planner_risk_top1_platt', 'risk_top1_platt'],
    'top1:raw': ['planner_risk_top1_proxy', 'risk_top1_raw'],
}
for key, cols in legacy_candidates.items():
    if key in available_risk_cols:
        continue
    for c in cols:
        if c in eval_df.columns:
            available_risk_cols[key] = c
            break

if len(available_risk_cols) == 0:
    raise RuntimeError('No usable risk columns found. Run cross_signal_decision_audit_colab.ipynb first.')

# Ensure default controller variant exists.
def _resolve_variant_key(pref_key):
    pref = str(pref_key).strip()
    if pref in available_risk_cols:
        return pref
    # fallback by signal with platt->raw preference
    if ':' in pref:
        signal = pref.split(':', 1)[0]
        for c in ['platt', 'iso', 'raw']:
            k = f'{signal}:{c}'
            if k in available_risk_cols:
                return k
    return sorted(available_risk_cols.keys())[0]

resolved_controller_variants = []
for k in PRIMARY_CONTROLLER_VARIANTS:
    kk = _resolve_variant_key(k)
    if kk not in resolved_controller_variants:
        resolved_controller_variants.append(kk)

eval_df['y_true'] = (pd.to_numeric(eval_df[FOCUS_LABEL], errors='coerce').fillna(0.0).to_numpy(dtype=float) > 0.5).astype(int)
cal_df['y_true'] = (pd.to_numeric(cal_df[FOCUS_LABEL], errors='coerce').fillna(0.0).to_numpy(dtype=float) > 0.5).astype(int)

eval_df['step_key'] = eval_df['planner_variant'].astype(str) + '::' + eval_df['scenario_id'] + '::' + eval_df['step_idx'].astype(str)
cal_df['step_key'] = cal_df['planner_variant'].astype(str) + '::' + cal_df['scenario_id'] + '::' + cal_df['step_idx'].astype(str)

comfort = np.zeros((len(eval_df),), dtype=float)
if 'max_abs_acc_h6' in eval_df.columns:
    comfort += np.abs(pd.to_numeric(eval_df['max_abs_acc_h6'], errors='coerce').fillna(0.0).to_numpy(dtype=float))
if 'max_abs_jerk_h6' in eval_df.columns:
    comfort += np.abs(pd.to_numeric(eval_df['max_abs_jerk_h6'], errors='coerce').fillna(0.0).to_numpy(dtype=float))
eval_df['comfort_cost'] = comfort

print('analysis_run_prefix (primary loaded) =', analysis_run_prefix)
print('audit_output_prefix (exports) =', AUDIT_OUTPUT_PREFIX)
print('loaded_run_count =', len(loaded_prefixes))
print('data_sources =', sorted(set([s for s in loaded_sources if s])))
print('split_source =', split_source)
print('rows(eval) =', len(eval_df), 'rows(cal) =', len(cal_df))
print('planner_variants =', sorted(eval_df['planner_variant'].astype(str).unique().tolist()))
print('resolved_controller_variants =', resolved_controller_variants)
print('available_risk_cols sample =', sorted(available_risk_cols.keys())[:15])

preview_cols = ['planner_variant', 'scenario_id', 'step_idx', 'shift_suite', 'y_true', 'progress_h6', 'comfort_cost']
for k in resolved_controller_variants[:2]:
    col = available_risk_cols.get(k, '')
    if col and col in eval_df.columns:
        eval_df[f'risk_{k}'] = _clip_prob(eval_df[col])
        preview_cols.append(f'risk_{k}')
preview_cols = [c for c in preview_cols if c in eval_df.columns]
display(eval_df[preview_cols].head(12))


## Step 6 - Controller Definitions and Evaluation Utilities


In [ ]:
import numpy as np
import pandas as pd

def _clip_prob(x):
    arr = np.asarray(pd.to_numeric(x, errors='coerce'), dtype=float)
    arr = np.where(np.isfinite(arr), arr, 0.5)
    return np.clip(arr, 1e-6, 1.0 - 1e-6)

def _risk_array(df, risk_col):
    return _clip_prob(df[risk_col])

def _select_chance_gate(step_df, risk_col, tau):
    # Chance-constrained style hard gate.
    risk = _risk_array(step_df, risk_col)
    accepted = risk <= float(tau)
    if np.any(accepted):
        sub = step_df.loc[accepted].copy()
        idx = sub['progress_h6'].astype(float).idxmax()
        fallback = 0
    else:
        idx = step_df[risk_col].astype(float).idxmin()
        fallback = 1
    row = step_df.loc[int(idx)]
    selected_risk = float(np.clip(float(row[risk_col]), 1e-6, 1.0 - 1e-6))
    return {
        'selected_idx': int(idx),
        'selected_risk': selected_risk,
        'accepted': int(selected_risk <= float(tau)),
        'fallback': int(fallback),
        'failure': int(row['y_true']),
        'progress': float(row['progress_h6']),
        'comfort': float(row['comfort_cost']),
        'selected_safe': int(int(row['y_true']) == 0),
    }

def _select_racp_rerank(step_df, risk_col, lam):
    # RACP-style reranking: progress - lambda * risk.
    r = _risk_array(step_df, risk_col)
    score = step_df['progress_h6'].astype(float).to_numpy(dtype=float) - float(lam) * r
    idx = int(step_df.index.to_numpy()[int(np.argmax(score))])
    row = step_df.loc[idx]
    selected_risk = float(np.clip(float(row[risk_col]), 1e-6, 1.0 - 1e-6))
    return {
        'selected_idx': int(idx),
        'selected_risk': selected_risk,
        'accepted': np.nan,  # defined later against reporting tau.
        'fallback': 0,
        'failure': int(row['y_true']),
        'progress': float(row['progress_h6']),
        'comfort': float(row['comfort_cost']),
        'selected_safe': int(int(row['y_true']) == 0),
    }

def _select_oracle_risk(step_df, tau):
    # Oracle-risk: true safety first; conservative tie-break by comfort.
    safe = step_df['y_true'].to_numpy(dtype=int) == 0
    if np.any(safe):
        sub = step_df.loc[safe].copy()
        idx = sub['comfort_cost'].astype(float).idxmin()
        fallback = 0
    else:
        idx = step_df['comfort_cost'].astype(float).idxmin()
        fallback = 1
    row = step_df.loc[int(idx)]
    oracle_risk = float(row['y_true'])
    return {
        'selected_idx': int(idx),
        'selected_risk': float(oracle_risk),
        'accepted': int(oracle_risk <= float(tau)),
        'fallback': int(fallback),
        'failure': int(row['y_true']),
        'progress': float(row['progress_h6']),
        'comfort': float(row['comfort_cost']),
        'selected_safe': int(int(row['y_true']) == 0),
    }

def _select_oracle_best(step_df, tau):
    # Oracle-best: among truly safe candidates, maximize progress.
    safe = step_df['y_true'].to_numpy(dtype=int) == 0
    if np.any(safe):
        sub = step_df.loc[safe].copy()
        idx = sub['progress_h6'].astype(float).idxmax()
        fallback = 0
    else:
        idx = step_df['progress_h6'].astype(float).idxmax()
        fallback = 1
    row = step_df.loc[int(idx)]
    oracle_risk = float(row['y_true'])
    return {
        'selected_idx': int(idx),
        'selected_risk': float(oracle_risk),
        'accepted': int(oracle_risk <= float(tau)),
        'fallback': int(fallback),
        'failure': int(row['y_true']),
        'progress': float(row['progress_h6']),
        'comfort': float(row['comfort_cost']),
        'selected_safe': int(int(row['y_true']) == 0),
    }

def _fit_conformal_tau(cal_subset, risk_col, tau_target, min_rows=80, min_accept=20):
    # Validation-driven effective tau: largest tau with empirical false-safe <= target and enough accepted mass.
    if cal_subset.empty or (risk_col not in cal_subset.columns):
        return float(tau_target), 'fallback:no_cal_subset'

    r = _risk_array(cal_subset, risk_col)
    y = cal_subset['y_true'].to_numpy(dtype=int)
    ok = np.isfinite(r)
    r = r[ok]
    y = y[ok]
    if r.size < int(min_rows):
        return float(tau_target), 'fallback:low_cal_rows'

    grid = np.asarray(TAU_SWEEP_VALUES, dtype=float)
    best_tau = None
    for t in sorted(grid.tolist()):
        accepted = r <= float(t)
        n_acc = int(np.sum(accepted))
        if n_acc < int(min_accept):
            continue
        fs = float(np.mean(y[accepted]))
        if np.isfinite(fs) and (fs <= float(tau_target)):
            best_tau = float(t)

    if best_tau is None:
        return float(min(grid)), 'fallback:no_tau_meets_target'
    return float(best_tau), 'ok'

def _evaluate_controller(df, controller_spec, tau):
    mode = str(controller_spec.get('mode', 'chance'))
    risk_col = str(controller_spec.get('risk_col', ''))
    lam = float(controller_spec.get('lambda', 0.0))
    tau_eff = float(controller_spec.get('tau_effective', tau))

    rows = []
    for (planner_variant, sid, step_idx, shift_suite), step_df in df.groupby(['planner_variant', 'scenario_id', 'step_idx', 'shift_suite'], sort=False):
        if mode == 'chance':
            out = _select_chance_gate(step_df, risk_col=risk_col, tau=float(tau))
            decision_tau = float(tau)
        elif mode == 'conformal':
            out = _select_chance_gate(step_df, risk_col=risk_col, tau=float(tau_eff))
            decision_tau = float(tau_eff)
        elif mode == 'racp':
            out = _select_racp_rerank(step_df, risk_col=risk_col, lam=float(lam))
            # Audit selected action against reporting tau.
            out['accepted'] = int(out['selected_risk'] <= float(tau))
            decision_tau = float(tau)
        elif mode == 'oracle-risk':
            out = _select_oracle_risk(step_df, tau=float(tau))
            decision_tau = float(tau)
        elif mode == 'oracle-best':
            out = _select_oracle_best(step_df, tau=float(tau))
            decision_tau = float(tau)
        else:
            raise ValueError(f'Unknown controller mode: {mode}')

        feasible = np.nan
        if risk_col and (risk_col in step_df.columns):
            feasible = int(np.any(_risk_array(step_df, risk_col) <= float(decision_tau)))

        rows.append({
            'planner_variant': str(planner_variant),
            'scenario_id': str(sid),
            'step_idx': int(step_idx),
            'shift_suite': str(shift_suite),
            'controller': str(controller_spec.get('name', mode)),
            'mode': mode,
            'signal_variant': str(controller_spec.get('variant_key', 'oracle')),
            'risk_col': risk_col,
            'tau': float(tau),
            'tau_effective': float(tau_eff),
            'decision_tau': float(decision_tau),
            **out,
            'safe_candidate_exists': int(np.any(step_df['y_true'].to_numpy(dtype=int) == 0)),
            'feasible_set': feasible,
        })

    step_df = pd.DataFrame(rows)
    if step_df.empty:
        return step_df, {}

    accepted = step_df['accepted'].to_numpy(dtype=float)
    accepted_mask = np.isfinite(accepted) & (accepted > 0.5)
    rejected_mask = np.isfinite(accepted) & (accepted <= 0.5)
    y = step_df['failure'].to_numpy(dtype=float)

    metrics = {
        'n_steps': int(len(step_df)),
        'n_scenarios': int(step_df['scenario_id'].nunique()),
        'failure_rate': float(np.mean(y)),
        'progress_mean': float(np.mean(step_df['progress'].to_numpy(dtype=float))),
        'accept_rate': float(np.mean(accepted_mask)) if len(step_df) else np.nan,
        'false_safe': float(np.mean(y[accepted_mask])) if int(np.sum(accepted_mask)) > 0 else np.nan,
        'safe_reject': float(np.mean((y <= 0.5)[rejected_mask])) if int(np.sum(rejected_mask)) > 0 else np.nan,
        'feasible_set_rate': float(np.nanmean(step_df['feasible_set'].to_numpy(dtype=float))) if 'feasible_set' in step_df.columns else np.nan,
        'fallback_rate': float(np.mean(step_df['fallback'].to_numpy(dtype=float))),
        'safe_candidate_rate': float(np.mean(step_df['safe_candidate_exists'].to_numpy(dtype=float))),
    }
    return step_df, metrics

def _bootstrap_ci(df_steps, n_boot=0, seed=17):
    n_boot = int(max(0, int(n_boot)))
    if (n_boot <= 0) or df_steps.empty:
        return {}
    # Reset index so bootstrap uses positional rows safely after filtering.
    work = df_steps.reset_index(drop=True)
    sid = work['scenario_id'].astype(str)
    groups = {k: np.asarray(v, dtype=int) for k, v in work.groupby(sid, sort=False).indices.items()}
    keys = list(groups.keys())
    if len(keys) <= 1:
        return {}

    rng = np.random.default_rng(int(seed))
    vals = []
    for _ in range(n_boot):
        picked = rng.integers(0, len(keys), size=len(keys))
        idx = np.concatenate([groups[keys[int(i)]] for i in picked], axis=0)
        sub = work.iloc[idx]

        accepted = sub['accepted'].to_numpy(dtype=float)
        accepted_mask = np.isfinite(accepted) & (accepted > 0.5)
        rejected_mask = np.isfinite(accepted) & (accepted <= 0.5)
        y = sub['failure'].to_numpy(dtype=float)

        vals.append({
            'failure_rate': float(np.mean(y)),
            'progress_mean': float(np.mean(sub['progress'].to_numpy(dtype=float))),
            'accept_rate': float(np.mean(accepted_mask)) if len(sub) else np.nan,
            'false_safe': float(np.mean(y[accepted_mask])) if int(np.sum(accepted_mask)) > 0 else np.nan,
            'safe_reject': float(np.mean((y <= 0.5)[rejected_mask])) if int(np.sum(rejected_mask)) > 0 else np.nan,
            'feasible_set_rate': float(np.nanmean(sub['feasible_set'].to_numpy(dtype=float))) if 'feasible_set' in sub.columns else np.nan,
            'fallback_rate': float(np.mean(sub['fallback'].to_numpy(dtype=float))),
        })

    out = {}
    for k in vals[0].keys():
        arr = np.asarray([x.get(k, np.nan) for x in vals], dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size == 0:
            continue
        out[f'{k}_ci_low'] = float(np.quantile(arr, 0.025))
        out[f'{k}_ci_high'] = float(np.quantile(arr, 0.975))
    return out


## Step 7 - Run Controller Comparisons and Tau Sweep


In [ ]:
import numpy as np
import pandas as pd

all_step_rows = []
all_metric_rows = []
conformal_rows = []

planners = sorted(eval_df['planner_variant'].astype(str).unique().tolist())

# Build controller specs.
controller_specs = []
for variant_key in resolved_controller_variants:
    risk_col = available_risk_cols.get(variant_key, '')
    if not risk_col:
        continue

    # Current/chance-constrained controller.
    controller_specs.append({
        'name': f'chance:{variant_key}',
        'mode': 'chance',
        'variant_key': variant_key,
        'risk_col': risk_col,
    })

    # RACP-style lambda sweep.
    for lam in RACP_LAMBDA_VALUES:
        controller_specs.append({
            'name': f'racp:{variant_key}:lam={lam:g}',
            'mode': 'racp',
            'variant_key': variant_key,
            'risk_col': risk_col,
            'lambda': float(lam),
        })

# Keep legacy label for compatibility if combo:platt exists.
for spec in controller_specs:
    if spec['name'] == 'chance:combo:platt':
        spec['name'] = 'current'
        break

# Oracle controls (signal-independent).
controller_specs.append({'name': 'oracle-risk', 'mode': 'oracle-risk', 'variant_key': 'oracle', 'risk_col': ''})
controller_specs.append({'name': 'oracle-best', 'mode': 'oracle-best', 'variant_key': 'oracle', 'risk_col': ''})

# Fit conformal effective taus per planner and variant.
conformal_tau_map = {}
for planner in planners:
    planner_cal = cal_df[cal_df['planner_variant'].astype(str).eq(str(planner))].copy()
    for variant_key in resolved_controller_variants:
        risk_col = available_risk_cols.get(variant_key, '')
        if not risk_col:
            continue

        if CONFORMAL_USE_SHIFT_LOCAL and ('shift_suite' in planner_cal.columns):
            shifts = sorted(planner_cal['shift_suite'].astype(str).unique().tolist())
        else:
            shifts = ['_global_']

        for shift in shifts:
            if shift == '_global_':
                subset = planner_cal
            else:
                subset = planner_cal[planner_cal['shift_suite'].astype(str).eq(str(shift))]

            tau_eff, reason = _fit_conformal_tau(
                subset,
                risk_col=risk_col,
                tau_target=float(RISK_BUDGET_TAU),
                min_rows=CONFORMAL_MIN_CAL_ROWS,
                min_accept=CONFORMAL_MIN_ACCEPT,
            )
            conformal_tau_map[(str(planner), str(variant_key), str(shift))] = float(tau_eff)
            conformal_rows.append({
                'planner_variant': str(planner),
                'variant_key': str(variant_key),
                'shift_suite': str(shift),
                'risk_col': risk_col,
                'tau_target': float(RISK_BUDGET_TAU),
                'tau_effective': float(tau_eff),
                'status': str(reason),
                'n_cal_rows': int(len(subset)),
                'pos_rate_cal': float(np.mean(subset['y_true'].to_numpy(dtype=float))) if len(subset) else np.nan,
            })

# Add conformal controllers.
for variant_key in resolved_controller_variants:
    risk_col = available_risk_cols.get(variant_key, '')
    if not risk_col:
        continue
    controller_specs.append({
        'name': f'conformal:{variant_key}',
        'mode': 'conformal',
        'variant_key': variant_key,
        'risk_col': risk_col,
    })

for planner in planners:
    planner_eval = eval_df[eval_df['planner_variant'].astype(str).eq(str(planner))].copy()
    if planner_eval.empty:
        continue
    shifts = sorted(planner_eval['shift_suite'].astype(str).unique().tolist())

    for shift_suite in shifts:
        shift_df = planner_eval[planner_eval['shift_suite'].astype(str).eq(str(shift_suite))].copy()
        if shift_df.empty:
            continue

        for spec in controller_specs:
            mode = str(spec.get('mode', 'chance'))
            variant_key = str(spec.get('variant_key', 'oracle'))

            if mode == 'conformal':
                tau_grid = [float(RISK_BUDGET_TAU)]
            elif mode == 'racp':
                tau_grid = [float(RISK_BUDGET_TAU)]
            elif mode in ('oracle-risk', 'oracle-best'):
                tau_grid = [float(RISK_BUDGET_TAU)]
            else:
                tau_grid = TAU_SWEEP_VALUES

            for tau in tau_grid:
                local_spec = dict(spec)
                if mode == 'conformal':
                    key_shift = str(shift_suite) if CONFORMAL_USE_SHIFT_LOCAL else '_global_'
                    tau_eff = conformal_tau_map.get((str(planner), str(variant_key), key_shift), float(RISK_BUDGET_TAU))
                    local_spec['tau_effective'] = float(tau_eff)
                else:
                    local_spec['tau_effective'] = float(tau)

                step_trace, metrics = _evaluate_controller(shift_df, controller_spec=local_spec, tau=float(tau))
                if step_trace.empty:
                    continue

                ci = _bootstrap_ci(step_trace, n_boot=BOOTSTRAP_SAMPLES, seed=BOOTSTRAP_SEED)

                gate_reasons = []
                if int(metrics.get('n_steps', 0)) < 40:
                    gate_reasons.append('low_steps')
                if int(step_trace['failure'].sum()) < 10:
                    gate_reasons.append('low_failures')
                if int(np.nansum(step_trace['accepted'].to_numpy(dtype=float) > 0.5)) < 20:
                    gate_reasons.append('low_accept_count')

                all_metric_rows.append({
                    'planner_variant': str(planner),
                    'shift_suite': str(shift_suite),
                    'domain': 'nominal' if str(shift_suite) == 'nominal_clean' else 'shifted',
                    'controller': str(local_spec.get('name', mode)),
                    'mode': mode,
                    'signal_variant': str(variant_key),
                    'risk_col': str(local_spec.get('risk_col', '')),
                    'lambda': float(local_spec.get('lambda', np.nan)) if mode == 'racp' else np.nan,
                    'tau': float(tau),
                    'tau_effective': float(local_spec.get('tau_effective', tau)),
                    **metrics,
                    **ci,
                    'status': 'ok' if len(gate_reasons) == 0 else 'inconclusive',
                    'inconclusive_reason': ';'.join(gate_reasons),
                })

                step_trace = step_trace.copy()
                step_trace['domain'] = 'nominal' if str(shift_suite) == 'nominal_clean' else 'shifted'
                step_trace['status'] = 'ok' if len(gate_reasons) == 0 else 'inconclusive'
                step_trace['inconclusive_reason'] = ';'.join(gate_reasons)
                all_step_rows.append(step_trace)

ctrl_step_df = pd.concat(all_step_rows, ignore_index=True) if len(all_step_rows) else pd.DataFrame()
ctrl_metrics_df = pd.DataFrame(all_metric_rows)
conformal_df = pd.DataFrame(conformal_rows)

if ctrl_metrics_df.empty:
    raise RuntimeError('No controller metrics were generated.')

at_budget_df = (
    ctrl_metrics_df.assign(_dist=(ctrl_metrics_df['tau'] - float(RISK_BUDGET_TAU)).abs())
    .sort_values('_dist')
    .groupby(['planner_variant', 'shift_suite', 'controller'], as_index=False)
    .first()
    .drop(columns=['_dist'])
    .sort_values(['planner_variant', 'shift_suite', 'controller'])
    .reset_index(drop=True)
)

# Bottleneck diagnosis per planner on nominal (fallback all).
diag_rows = []
for planner in planners:
    sub = at_budget_df[at_budget_df['planner_variant'].astype(str).eq(str(planner))].copy()
    focus_diag_df = sub[sub['shift_suite'].astype(str).eq('nominal_clean')].copy()
    if focus_diag_df.empty:
        focus_diag_df = sub.copy()

    diag = {
        'planner_variant': str(planner),
        'signal_or_calibration_bottleneck': np.nan,
        'controller_rule_bottleneck': np.nan,
        'candidate_quality_bottleneck': np.nan,
    }
    diag_evidence = {}

    def _row_for(name):
        r = focus_diag_df[focus_diag_df['controller'].astype(str).eq(name)]
        return r.iloc[0] if not r.empty else None

    r_cur = _row_for('current')
    if r_cur is None:
        # fallback to chance:combo:platt
        r_cur = _row_for('chance:combo:platt')
    r_orisk = _row_for('oracle-risk')
    r_obest = _row_for('oracle-best')

    if (r_cur is not None) and (r_orisk is not None):
        fail_gap = float(r_cur['failure_rate']) - float(r_orisk['failure_rate'])
        diag['signal_or_calibration_bottleneck'] = int(fail_gap > 0.02)
        diag_evidence['signal_fail_gap_current_minus_oracle_risk'] = fail_gap

    if (r_orisk is not None) and (r_obest is not None):
        prog_gap = float(r_obest['progress_mean']) - float(r_orisk['progress_mean'])
        fail_gap_ob = float(r_obest['failure_rate']) - float(r_orisk['failure_rate'])
        diag['controller_rule_bottleneck'] = int((prog_gap > 0.02) and (fail_gap_ob <= 0.01))
        diag_evidence['rule_progress_gap_oracle_best_minus_oracle_risk'] = prog_gap
        diag_evidence['rule_failure_gap_oracle_best_minus_oracle_risk'] = fail_gap_ob

    if r_obest is not None:
        safe_rate = float(r_obest['safe_candidate_rate']) if np.isfinite(float(r_obest['safe_candidate_rate'])) else np.nan
        fail_rate = float(r_obest['failure_rate'])
        diag['candidate_quality_bottleneck'] = int((np.isfinite(safe_rate) and safe_rate < 0.75) or (fail_rate > 0.15))
        diag_evidence['oracle_best_safe_candidate_rate'] = safe_rate
        diag_evidence['oracle_best_failure_rate'] = fail_rate

    diag_rows.append({
        **diag,
        **diag_evidence,
        'note': '1 indicates likely bottleneck present under this offline candidate-set audit.',
    })

diagnosis_df = pd.DataFrame(diag_rows)

print('planner_variants =', planners)
print('controller_specs =', len(controller_specs))
print('ctrl_metrics_df rows =', len(ctrl_metrics_df))
print('ctrl_step_df rows =', len(ctrl_step_df))
print('inconclusive metric rows =', int((ctrl_metrics_df['status'] == 'inconclusive').sum()))

display(conformal_df.head(30))
display(at_budget_df.head(40))
display(diagnosis_df)


## Step 8 - Progress vs Failure Tradeoff and Controller Comparison Plots


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

figure_paths = {}

# Plot 1: Progress vs failure tradeoff for major controller families.
fig, ax = plt.subplots(1, 1, figsize=(9, 6))
plot_specs = [
    ('current', 'tab:blue'),
    ('chance:combo:raw', 'tab:gray'),
    ('chance:combo:platt', 'tab:cyan'),
    ('conformal:combo:platt', 'tab:purple'),
    ('oracle-risk', 'tab:red'),
    ('oracle-best', 'tab:green'),
]
for name, color in plot_specs:
    sub = ctrl_metrics_df[ctrl_metrics_df['controller'].astype(str).eq(name)].copy()
    if sub.empty:
        continue
    sub = (
        sub.groupby('tau', as_index=False)
        .agg(failure_rate=('failure_rate', 'mean'), progress_mean=('progress_mean', 'mean'))
        .sort_values('tau')
    )
    ax.plot(sub['progress_mean'], sub['failure_rate'], marker='o', linewidth=1.7, label=name, color=color)

# Add RACP lambda curve (at fixed tau, lambda sweep points).
racp_sub = ctrl_metrics_df[ctrl_metrics_df['controller'].astype(str).str.contains('^racp:combo:platt:lam=', regex=True)].copy()
if not racp_sub.empty:
    racp_sub = racp_sub.sort_values('lambda')
    ax.plot(racp_sub['progress_mean'], racp_sub['failure_rate'], marker='s', linewidth=1.7, label='racp:combo:platt (lambda sweep)', color='tab:orange')

ax.set_title('Progress vs Failure Tradeoff (controller families)')
ax.set_xlabel('mean progress')
ax.set_ylabel('failure rate')
ax.grid(alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()
p = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_tradeoff.png'
fig.savefig(p, dpi=170, bbox_inches='tight')
figure_paths['oracle_bottleneck_tradeoff'] = p
plt.show()

# Plot 2: Budget diagnostics near tau=0.2 for selected controllers.
selected = at_budget_df[at_budget_df['controller'].astype(str).isin(['current', 'chance:combo:raw', 'chance:combo:platt', 'conformal:combo:platt', 'oracle-risk', 'oracle-best'])].copy()
if not selected.empty:
    metric_cols = ['accept_rate', 'false_safe', 'safe_reject', 'feasible_set_rate', 'fallback_rate']
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(4.6 * len(metric_cols), 5))
    for ax, metric in zip(axes, metric_cols):
        pivot = selected.pivot_table(index='planner_variant', columns='controller', values=metric, aggfunc='mean')
        cols = [c for c in ['current', 'chance:combo:raw', 'chance:combo:platt', 'conformal:combo:platt', 'oracle-risk', 'oracle-best'] if c in pivot.columns]
        pivot = pivot.loc[:, cols]
        pivot.plot(kind='bar', ax=ax)
        ax.set_title(metric)
        ax.set_xlabel('planner_variant')
        ax.set_ylabel(metric)
        if metric == 'false_safe':
            ax.axhline(float(RISK_BUDGET_TAU), color='k', linestyle='--', linewidth=1)
        ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    p = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_budget_metrics.png'
    fig.savefig(p, dpi=170, bbox_inches='tight')
    figure_paths['oracle_bottleneck_budget_metrics'] = p
    plt.show()

# Plot 3: RACP lambda sweep details (combo:platt if present).
racp_lambda = ctrl_metrics_df[ctrl_metrics_df['controller'].astype(str).str.contains('^racp:combo:platt:lam=', regex=True)].copy()
if not racp_lambda.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
    ag = (
        racp_lambda.groupby('lambda', as_index=False)
        .agg(failure_rate=('failure_rate', 'mean'), progress_mean=('progress_mean', 'mean'))
        .sort_values('lambda')
    )
    axes[0].plot(ag['lambda'], ag['failure_rate'], marker='o')
    axes[0].set_title('RACP-style failure vs lambda')
    axes[0].set_xlabel('lambda')
    axes[0].set_ylabel('failure_rate')
    axes[0].grid(alpha=0.25)

    axes[1].plot(ag['lambda'], ag['progress_mean'], marker='o')
    axes[1].set_title('RACP-style progress vs lambda')
    axes[1].set_xlabel('lambda')
    axes[1].set_ylabel('progress_mean')
    axes[1].grid(alpha=0.25)

    fig.tight_layout()
    p = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_racp_lambda.png'
    fig.savefig(p, dpi=170, bbox_inches='tight')
    figure_paths['oracle_bottleneck_racp_lambda'] = p
    plt.show()

print('figure_paths =', figure_paths)


## Step 9 - Export Artifacts + Contract Manifest


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

metrics_path = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_metrics.csv'
step_path = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_step_trace.csv'
budget_path = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_at_budget.csv'
diagnosis_path = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_diagnosis.csv'
conformal_path = f'{AUDIT_OUTPUT_PREFIX}_oracle_bottleneck_conformal_tau.csv'

ctrl_metrics_df.to_csv(metrics_path, index=False)
ctrl_step_df.to_csv(step_path, index=False)
at_budget_df.to_csv(budget_path, index=False)
diagnosis_df.to_csv(diagnosis_path, index=False)
if 'conformal_df' in globals() and isinstance(conformal_df, pd.DataFrame) and (not conformal_df.empty):
    conformal_df.to_csv(conformal_path, index=False)

artifact_paths = {
    'oracle_bottleneck_metrics': metrics_path,
    'oracle_bottleneck_step_trace': step_path,
    'oracle_bottleneck_at_budget': budget_path,
    'oracle_bottleneck_diagnosis': diagnosis_path,
    **figure_paths,
}
if 'conformal_df' in globals() and isinstance(conformal_df, pd.DataFrame) and (not conformal_df.empty):
    artifact_paths['oracle_bottleneck_conformal_tau'] = conformal_path

extra = {
    'analysis_run_prefix': str(analysis_run_prefix),
    'audit_output_prefix': str(AUDIT_OUTPUT_PREFIX),
    'focus_label': str(FOCUS_LABEL),
    'risk_budget_tau': float(RISK_BUDGET_TAU),
    'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
    'metrics_rows': int(len(ctrl_metrics_df)),
    'step_rows': int(len(ctrl_step_df)),
    'inconclusive_rows': int((ctrl_metrics_df['status'] == 'inconclusive').sum()) if not ctrl_metrics_df.empty else 0,
    'planner_count': int(ctrl_metrics_df['planner_variant'].nunique()) if 'planner_variant' in ctrl_metrics_df.columns and (not ctrl_metrics_df.empty) else 0,
    'controller_count': int(ctrl_metrics_df['controller'].nunique()) if not ctrl_metrics_df.empty else 0,
    'figure_count': int(len(figure_paths)),
}

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage='oracle_bottleneck_completed',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='oracle_bottleneck_completed',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)

print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
for k, v in sorted(artifact_paths.items()):
    print('-', k, '->', v)


## Step 10 - Final Summary


In [ ]:
import numpy as np

print('=== Final Summary (Oracle Bottleneck Analysis) ===')
print('focus_label =', FOCUS_LABEL, '| tau =', RISK_BUDGET_TAU)

nominal = at_budget_df[at_budget_df['shift_suite'].astype(str).eq('nominal_clean')].copy()
if nominal.empty:
    nominal = at_budget_df.copy()

planners = sorted(nominal['planner_variant'].astype(str).unique().tolist())
for planner in planners:
    subp = nominal[nominal['planner_variant'].astype(str).eq(str(planner))]
    if subp.empty:
        continue
    print(f"\n[planner_variant: {planner}]")
    for controller in ['current', 'chance:combo:raw', 'chance:combo:platt', 'conformal:combo:platt', 'oracle-risk', 'oracle-best']:
        sub = subp[subp['controller'].astype(str).eq(controller)]
        if sub.empty:
            continue
        r = sub.iloc[0]
        fs = float(r['false_safe']) if np.isfinite(float(r['false_safe'])) else float('nan')
        sr = float(r['safe_reject']) if np.isfinite(float(r['safe_reject'])) else float('nan')
        print(f"  [{controller}] failure={float(r['failure_rate']):.4f} progress={float(r['progress_mean']):.4f} "
              f"accept={float(r['accept_rate']):.4f} false_safe={fs:.4f} safe_reject={sr:.4f} "
              f"feasible={float(r['feasible_set_rate']):.4f} fallback={float(r['fallback_rate']):.4f}")

if not diagnosis_df.empty:
    print('\nLikely bottlenecks by planner (1=yes, 0=no, nan=inconclusive):')
    cols = [
        'planner_variant',
        'signal_or_calibration_bottleneck',
        'controller_rule_bottleneck',
        'candidate_quality_bottleneck',
        'oracle_best_safe_candidate_rate',
        'oracle_best_failure_rate',
    ]
    cols = [c for c in cols if c in diagnosis_df.columns]
    display(diagnosis_df.loc[:, cols])

if 'conformal_df' in globals() and isinstance(conformal_df, pd.DataFrame) and (not conformal_df.empty):
    print('\nConformal effective tau summary (head):')
    display(conformal_df.head(20))

print('\nInterpretation guideline:')
print('- chance:* isolates threshold-rule behavior for each risk formulation.')
print('- racp:* isolates smooth risk-penalized reranking via lambda sweep.')
print('- conformal:* tests validation-calibrated effective tau against fixed tau controllers.')
print('- oracle-* provides candidate-set upper bounds for bottleneck diagnosis.')
